# 📦 Enterprise Supply Chain Diagnostic: Delivery SLA Breach, Bottlenecks & Concession Analytics

**Author:** Data Analytics & Operations Research Team  
**Target Role:** Senior Supply Chain / Business Intelligence Analyst  
**Dataset:** Olist Logistics Network (100,000+ Real-World Orders)  

---

## 🎯 Executive Problem Statement
In e-commerce fulfillment, failing to meet the **Promised Delivery Date (PDD)** imposes a severe double penalty:
1. **Direct Operational Loss:** Customer concessions (freight refunds, appeasement credits, support overhead).
2. **Long-Term Revenue Loss:** Customer churn and review decay (Net Promoter Score collapse).

This notebook conducts a deep diagnostic across **7 critical analytical dimensions** to identify root causes, quantify financial exposure, and design data-backed fulfillment interventions.

## 🛠️ Environment Setup & Data Ingestion

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Professional styling standards
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

DATA_PATH = os.path.join("..", "data", "orders_fulfillment_bi.csv")
df = pd.read_csv(DATA_PATH)

# Convert timestamp strings
date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

print(f"Logistics telemetry records loaded: {len(df):,}")
df.head(3)

## 📊 Module 1: Baseline SLA Compliance & Network Defect Density (DPMO)
Tracking actual delivery date against customer Promised Delivery Date (PDD) and calculating Defects Per Million Opportunities (DPMO).

In [ ]:
total_orders = len(df)
breached_orders = df['is_sla_breach'].sum()
ontime_orders = total_orders - breached_orders
breach_rate = (breached_orders / total_orders) * 100
dpmo = (breached_orders / total_orders) * 1_000_000

print(f"Total Delivered Shipments: {total_orders:,}")
print(f"On-Time Deliveries:        {ontime_orders:,} ({100 - breach_rate:.2f}%)")
print(f"SLA Breached Deliveries:   {breached_orders:,} ({breach_rate:.2f}%)")
print(f"Network Defect Density:    {dpmo:,.0f} DPMO (Target: < 20,000 DPMO)")

# Visualization: SLA Adherence Donut Chart
fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    [ontime_orders, breached_orders],
    labels=['On-Time (Compliant)', 'SLA Breached (Late)'],
    autopct='%1.1f%%',
    startangle=140,
    colors=['#10b981', '#ef4444'],
    explode=(0, 0.08),
    wedgeprops=dict(width=0.45, edgecolor='w')
)
plt.title('Module 1: Network-Wide Delivery SLA Adherence')
plt.show()

## 🚚 Module 2: Multi-Stage Latency Decomposition (Seller vs Carrier)
Decomposing fulfillment duration into:
1. **Seller Warehouse Dispatch Lag** (Order approval $\rightarrow$ Courier pickup)
2. **Carrier Transit Lag** (Courier pickup $\rightarrow$ Customer doorstep)

In [ ]:
stage_summary = df.groupby('is_sla_breach')[['seller_dispatch_days', 'carrier_transit_days', 'actual_delivery_days']].mean().reset_index()
stage_summary['Status'] = stage_summary['is_sla_breach'].map({0: 'On-Time Deliveries', 1: 'SLA Breached Deliveries'})

# Calculate carrier share of delay
breached_row = stage_summary[stage_summary['is_sla_breach'] == 1].iloc[0]
carrier_share = (breached_row['carrier_transit_days'] / breached_row['actual_delivery_days']) * 100

print(f"Carrier transit accounts for {carrier_share:.1f}% of total delivery latency in breached orders!")

# Stacked Stage Latency Bar Chart
ax = stage_summary.plot(
    x='Status',
    y=['seller_dispatch_days', 'carrier_transit_days'],
    kind='bar',
    stacked=True,
    color=['#38bdf8', '#f97316'],
    figsize=(9, 5),
    edgecolor='w'
)
plt.title('Module 2: Stage Latency Breakdown (Seller Packing vs Carrier Transit)')
plt.ylabel('Average Days')
plt.xticks(rotation=0)
plt.legend(['Seller Dispatch Lag (Warehouse)', 'Carrier Transit Lag (Logistics)'])
plt.show()

## 🗺️ Module 3: Geographic Topology (Intra-State vs Inter-State Route Matrix)
Evaluating delivery performance when the customer and seller are in the **same state** vs **different states across the country**.

In [ ]:
df['is_interstate'] = (df['seller_state'] != df['customer_state']).astype(int)

geo_perf = df.groupby('is_interstate').agg(
    total_shipments=('order_id', 'count'),
    sla_breach_rate=('is_sla_breach', lambda x: x.mean() * 100),
    avg_transit_days=('carrier_transit_days', 'mean'),
    avg_freight_cost=('freight_value', 'mean')
).reset_index()
geo_perf['Route Type'] = geo_perf['is_interstate'].map({0: 'Intra-State (Same State)', 1: 'Inter-State (Cross-Country)'})

print(geo_perf[['Route Type', 'total_shipments', 'sla_breach_rate', 'avg_transit_days', 'avg_freight_cost']])

# Visualization
fig, ax1 = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=geo_perf, x='Route Type', y='sla_breach_rate', palette=['#10b981', '#ef4444'], ax=ax1)
ax1.set_ylabel('SLA Breach Rate (%)', fontweight='bold', color='#1e293b')
ax1.set_title('Module 3: Intra-State vs Inter-State Delivery Failure Rate')
plt.show()

## 📦 Module 4: Product Category, Weight & Freight-to-Price Sensitivity
Analyzing how freight shipping costs and delivery delays vary across shipping value tiers.

In [ ]:
df['freight_tier'] = pd.qcut(df['freight_value'], q=3, labels=['Low Freight (<$15)', 'Medium Freight ($15-$25)', 'High Freight (>$25)'])

freight_analysis = df.groupby('freight_tier').agg(
    order_count=('order_id', 'count'),
    breach_rate=('is_sla_breach', lambda x: x.mean() * 100),
    avg_transit_days=('carrier_transit_days', 'mean'),
    avg_review_score=('review_score', 'mean')
).reset_index()

print(freight_analysis)

# Visualization
plt.figure(figsize=(9, 4.5))
sns.barplot(data=freight_analysis, x='freight_tier', y='avg_transit_days', palette='Blues_d')
plt.title('Module 4: Transit Delay by Freight Cost / Weight Tier')
plt.ylabel('Average Carrier Transit Days')
plt.show()

## ⏱️ Module 5: Seller Fulfillment Performance & SLA Scorecard
Segmenting sellers into fulfillment speed tiers: **Gold (<24h)**, **Silver (24-72h)**, and **At-Risk (>72h)**.

In [ ]:
def tier_seller(dispatch_days):
    if dispatch_days <= 1.0:
        return 'Tier 1: Gold (< 24h Handover)'
    elif dispatch_days <= 3.0:
        return 'Tier 2: Silver (24h - 72h Handover)'
    else:
        return 'Tier 3: At-Risk (> 72h Handover)'

df['seller_tier'] = df['seller_dispatch_days'].apply(tier_seller)
seller_scorecard = df.groupby('seller_tier').agg(
    total_orders=('order_id', 'count'),
    sla_breach_rate=('is_sla_breach', lambda x: x.mean() * 100),
    avg_dispatch_days=('seller_dispatch_days', 'mean')
).reset_index()

print(seller_scorecard)

# Visualization
plt.figure(figsize=(8, 4.5))
sns.barplot(data=seller_scorecard, x='seller_tier', y='sla_breach_rate', palette='viridis')
plt.title('Module 5: Impact of Seller Packing Speed on Final SLA Breach')
plt.ylabel('SLA Breach Rate (%)')
plt.xticks(rotation=15)
plt.show()

## 📈 Module 6: Seasonality & Holiday Surge Stress Test (Peak Shock)
Tracking monthly order volume vs delivery breach rate to detect carrier network capacity choke points.

In [ ]:
monthly_trend = df.groupby('order_month').agg(
    orders=('order_id', 'count'),
    breach_rate=('is_sla_breach', lambda x: x.mean() * 100),
    avg_delay=('delivery_variance_days', lambda x: x[x > 0].mean())
).reset_index()

print(monthly_trend)

# Dual Axis Trend Plot
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(monthly_trend['order_month'], monthly_trend['orders'], color='#94a3b8', alpha=0.6, label='Order Volume')
ax1.set_ylabel('Order Volume', color='#475569', fontweight='bold')

ax2 = ax1.twinx()
ax2.plot(monthly_trend['order_month'], monthly_trend['breach_rate'], color='#dc2626', marker='o', linewidth=3, label='SLA Breach %')
ax2.set_ylabel('SLA Breach Rate (%)', color='#dc2626', fontweight='bold')

plt.title('Module 6: Monthly Logistics Capacity & SLA Breach Trajectory')
plt.show()

## 💔 Module 7: Customer Trust Collapse & Financial Concession Modeling
Modeling the correlation between delivery delay and customer review score decay (1-Star reviews) + estimating financial concession liabilities.

In [ ]:
# Review score comparison
review_impact = df.groupby('is_sla_breach').agg(
    avg_review_score=('review_score', 'mean'),
    pct_1_star=('review_score', lambda x: (x == 1).mean() * 100),
    pct_5_star=('review_score', lambda x: (x == 5).mean() * 100)
).reset_index()
review_impact['Status'] = review_impact['is_sla_breach'].map({0: 'On-Time Deliveries', 1: 'SLA Breached Deliveries'})

print(review_impact[['Status', 'avg_review_score', 'pct_1_star', 'pct_5_star']])

# Financial Concession Calculation
# Policy: Full freight fee refund on breached orders + $10 goodwill credit
breached_subset = df[df['is_sla_breach'] == 1]
total_freight_refund = breached_subset['freight_value'].sum()
goodwill_credits = len(breached_subset) * 10.0
total_concession_loss = total_freight_refund + goodwill_credits

print("\n" + "=" * 50)
print("FINANCIAL CONCESSION LIABILITY SUMMARY:")
print(f"Direct Freight Refund Loss:  ${total_freight_refund:,.2f}")
print(f"Goodwill Appeasement Credits: ${goodwill_credits:,.2f}")
print(f"Total Concession Exposure:   ${total_concession_loss:,.2f}")
print("=" * 50)

# Visualization: Review Score Decay
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(data=review_impact, x='Status', y='avg_review_score', palette=['#10b981', '#ef4444'], ax=ax)
ax.set_ylabel('Average Customer Rating (1-5 Stars)', fontweight='bold')
ax.set_title('Module 7: Customer Rating Decay on Breached Deliveries')
ax.set_ylim(0, 5.5)
plt.show()

## 🎯 Strategic Executive Recommendations
1. **Dynamic SLA Buffering (+3 Days):** Inject an algorithmic buffer for all inter-state routes from Southeast hubs to Northern/Northeastern destinations.
2. **Carrier SLA Performance Chargebacks:** Enforce contractual 15%/day freight penalties for carrier transit lags exceeding commitments.
3. **Proactive Pre-Breach Customer Communication:** Automatically notify customers and issue preemptive credits when parcels remain idle > 48h in transit hubs.